## Importando Bibliotecas e Carregando os Dados

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler

In [ ]:
caminho_atual = Path(".").resolve()
if caminho_atual.name == "notebooks":
    raiz_projeto = caminho_atual.parent
else:
    raiz_projeto = caminho_atual

df = pd.read_csv(raiz_projeto / "data" / "processed" / "2nd_test_features.csv")
print(f"Dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas")
display(df.head())

## 1. Remoção de Arquivos com Motor Desligado e Checagem de Inconsistências

In [ ]:
# Nos últimos 2 arquivos (idx 982 e 983) a máquina já havia quebrado e o motor foi desligado
print("Últimos 4 arquivos do experimento (notar queda abrupta do RMS em 982 e 983):")
display(df[["idx", "filename", "b1_rms", "b1_kurtosis"]].tail(4))

# Mantemos apenas os arquivos com o motor em operação (0 até 981)
df_ativo = df[df["idx"] <= 981].copy()
print(f"\nTotal de arquivos válidos em operação: {df_ativo.shape[0]}")

## 2. Seleção das Colunas (Features do Rolamento 1) e Checagem de Valores Ausentes

In [ ]:
colunas_meta = ["idx", "filename", "timestamp"]
colunas_b1 = [c for c in df_ativo.columns if c.startswith("b1_")]

df_b1 = df_ativo[colunas_meta + colunas_b1].copy()

nulos = pd.DataFrame({
    "Quantidade de Nulos": df_b1.isnull().sum(),
    "Porcentagem": (df_b1.isnull().mean() * 100).map("{:.2f}%".format)
})
print(f"Colunas selecionadas ({len(colunas_b1)} features):")
display(nulos)

## 3. Divisão Temporal dos Dados (Evitando *Data Leakage*)
Como se trata de uma série temporal *run-to-failure*, não podemos embaralhar os dados (`shuffle=False`). Seguimos o protocolo temporal:
- **Treino (`0 a 400`):** Início da vida do rolamento (100% saudável). Usado para treinar o Autoencoder.
- **Zona Cinza (`401 a 450`):** Intervalo de transição excluído para garantir separação limpa.
- **Validação (`451 a 530`):** Dados saudáveis usados apenas para calibrar o limiar (*threshold*) de alarme.
- **Teste (`531 a 981`):** Período onde surge e evolui a falha na pista externa.

In [ ]:
df_train = df_b1[(df_b1["idx"] >= 0) & (df_b1["idx"] <= 400)].copy()
df_val   = df_b1[(df_b1["idx"] >= 451) & (df_b1["idx"] <= 530)].copy()
df_test  = df_b1[(df_b1["idx"] >= 531) & (df_b1["idx"] <= 981)].copy()

print(f"Treino (Saudável):     {df_train.shape[0]} amostras (idx 0..400)")
print(f"Validação (Saudável):  {df_val.shape[0]} amostras  (idx 451..530)")
print(f"Teste (Degradação):    {df_test.shape[0]} amostras (idx 531..981)")

In [ ]:
# Visualizando a divisão temporal sobre a curva do RMS
plt.figure(figsize=(13, 4.5))
plt.plot(df_b1["idx"], df_b1["b1_rms"], color="black", lw=1.5, label="b1_rms")

plt.axvspan(0, 400, color="#27ae60", alpha=0.2, label="Treino (0..400)")
plt.axvspan(401, 450, color="gray", alpha=0.2, label="Zona Cinza (401..450)")
plt.axvspan(451, 530, color="#2980b9", alpha=0.2, label="Validação (451..530)")
plt.axvspan(531, 981, color="#c0392b", alpha=0.2, label="Teste / Falha (531..981)")

plt.title("Divisão Temporal do Dataset — Rolamento 1 (2nd_test)")
plt.xlabel("Índice do Arquivo")
plt.ylabel("RMS (g)")
plt.legend(loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Padronização (`StandardScaler`) e Salvamento dos Dados Processados
- O `StandardScaler` é ajustado (`fit`) **somente no conjunto de treino** para que nenhuma informação do futuro vaze para o modelo.

In [ ]:
pasta_processed = raiz_projeto / "data" / "processed"
pasta_processed.mkdir(parents=True, exist_ok=True)

scaler = StandardScaler()

df_train_scaled = df_train.copy()
df_val_scaled   = df_val.copy()
df_test_scaled  = df_test.copy()

# Ajusta apenas no treino e aplica em treino, validação e teste
df_train_scaled[colunas_b1] = scaler.fit_transform(df_train[colunas_b1])
df_val_scaled[colunas_b1]   = scaler.transform(df_val[colunas_b1])
df_test_scaled[colunas_b1]  = scaler.transform(df_test[colunas_b1])

# Salvando os arquivos para a etapa de modelagem
df_train_scaled.to_csv(pasta_processed / "train_scaled.csv", index=False)
df_val_scaled.to_csv(pasta_processed / "val_scaled.csv", index=False)
df_test_scaled.to_csv(pasta_processed / "test_scaled.csv", index=False)

print("Datasets padronizados e salvos com sucesso em data/processed/:")
print("- train_scaled.csv:", df_train_scaled.shape)
print("- val_scaled.csv:  ", df_val_scaled.shape)
print("- test_scaled.csv: ", df_test_scaled.shape)
display(df_train_scaled.head())